# 08 Kaggle Evidence Upgrade

Kaggle-first rerun notebook for the ElectroMacroDiff evidence-upgrade phase. This notebook keeps the frozen V5.3 branch untouched, writes all rerun outputs as sidecar artifacts, and refreshes the benchmark report plus dashboard data.

## 1. Setup

Upload the project zip as a Kaggle input dataset, then run this cell. It will unpack the repo, install the project requirements, and detect whether CUDA is available.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import zipfile

def run(command, cwd=None):
    print("RUN:", " ".join(str(part) for part in command))
    subprocess.run([str(part) for part in command], cwd=cwd, check=True)

input_zips = sorted(Path('/kaggle/input').rglob('*.zip'))
assert input_zips, 'Upload the ElectroMacroDiff project zip as a Kaggle input first.'
project_zip = input_zips[0]
print('Using zip:', project_zip)

run_root = Path('/kaggle/working/evidence_upgrade_run')
if run_root.exists():
    shutil.rmtree(run_root)
run_root.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(str(project_zip), str(run_root))

project_candidates = [run_root] + [path for path in run_root.rglob('*') if path.is_dir()]
project_candidates = [path for path in project_candidates if (path / 'scripts').exists() and (path / 'src').exists()]
assert project_candidates, 'Could not find the unpacked project root.'
BASE_PATH = next((path for path in project_candidates if path.name == 'EMD_V5_2_Hybrid'), project_candidates[0])
print('Project root:', BASE_PATH)

requirements = BASE_PATH / 'requirements_v5_2_hybrid_colab.txt'
if requirements.exists():
    run(['python', '-m', 'pip', 'install', '-q', '-r', str(requirements)])

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print({'device': DEVICE, 'cuda_available': torch.cuda.is_available()})

## 2. Logged Rerun

This generates a sidecar branch with attempt logging enabled and measures raw-attempt validity plus linker novelty without touching the frozen baseline files.

In [ ]:
gen_dir = BASE_PATH / '05_generated_candidates' / 'model_guided_macrocycle'
logged_generated = gen_dir / 'generated_v5_3_model_guided_macrocycles_kaggle_logged.csv'
logged_attempt = gen_dir / 'generated_v5_3_model_guided_macrocycles_kaggle_logged_attempt_log.csv'
logged_metrics = gen_dir / 'generated_v5_3_model_guided_macrocycles_kaggle_logged_metrics.csv'
logged_summary = gen_dir / 'generated_v5_3_model_guided_macrocycles_kaggle_logged_summary.json'
logged_novelty = gen_dir / 'v5_3_model_guided_linker_novelty_kaggle_logged.csv'
logged_gap = gen_dir / 'v5_3_model_guided_benchmark_gap_summary_kaggle_logged.json'

run([
    'python', str(BASE_PATH / 'scripts' / '17_generate_v5_3_model_guided_macrocycles.py'),
    '--base', str(BASE_PATH),
    '--device', DEVICE,
    '--seed-count', '250',
    '--max-products-per-seed', '12',
    '--skip-merged-output',
    '--output-csv', str(logged_generated),
    '--attempt-log-csv', str(logged_attempt),
    '--metrics-csv', str(logged_metrics),
    '--summary-json', str(logged_summary),
])

run([
    'python', str(BASE_PATH / 'scripts' / '20_measure_v5_3_benchmark_gaps.py'),
    '--base', str(BASE_PATH),
    '--generated-csv', str(logged_generated),
    '--attempt-log', str(logged_attempt),
    '--output-csv', str(logged_novelty),
    '--summary-json', str(logged_gap),
])

print(json.loads(logged_gap.read_text(encoding='utf-8')))

## 3. Diverse Rerun

This branch expands only the chain-compatible chemotypes already supported by the repo and enforces exact-linker novelty during generation.

In [ ]:
diverse_generated = gen_dir / 'generated_v5_3_model_guided_macrocycles_kaggle_diverse.csv'
diverse_attempt = gen_dir / 'generated_v5_3_model_guided_macrocycles_kaggle_diverse_attempt_log.csv'
diverse_metrics = gen_dir / 'generated_v5_3_model_guided_macrocycles_kaggle_diverse_metrics.csv'
diverse_summary = gen_dir / 'generated_v5_3_model_guided_macrocycles_kaggle_diverse_summary.json'
diverse_novelty = gen_dir / 'v5_3_model_guided_linker_novelty_kaggle_diverse.csv'
diverse_gap = gen_dir / 'v5_3_model_guided_benchmark_gap_summary_kaggle_diverse.json'

run([
    'python', str(BASE_PATH / 'scripts' / '17_generate_v5_3_model_guided_macrocycles.py'),
    '--base', str(BASE_PATH),
    '--device', DEVICE,
    '--seed-count', '250',
    '--max-products-per-seed', '12',
    '--skip-merged-output',
    '--chemotype-set', 'diverse',
    '--enforce-linker-novelty',
    '--novelty-mode', 'exact',
    '--output-csv', str(diverse_generated),
    '--attempt-log-csv', str(diverse_attempt),
    '--metrics-csv', str(diverse_metrics),
    '--summary-json', str(diverse_summary),
])

run([
    'python', str(BASE_PATH / 'scripts' / '20_measure_v5_3_benchmark_gaps.py'),
    '--base', str(BASE_PATH),
    '--generated-csv', str(diverse_generated),
    '--attempt-log', str(diverse_attempt),
    '--output-csv', str(diverse_novelty),
    '--summary-json', str(diverse_gap),
])

print(json.loads(diverse_gap.read_text(encoding='utf-8')))

## 4. SE(3) Scoring And Export

This scores auxiliary SE(3) geometry confidence for the ranked branch, refreshes the benchmark report and dashboard data, and bundles the sidecar outputs into one zip.

In [ ]:
ranking_input_candidates = [
    BASE_PATH / '08_final_ranking' / 'v5_3_model_guided_pocket_electronic_ranked_candidates.csv',
    BASE_PATH / '08_final_ranking' / 'v5_3_model_guided_ranked_candidates.csv',
]
ranking_input = next((path for path in ranking_input_candidates if path.exists()), None)
se3_checkpoint = BASE_PATH / '04_models_checkpoints' / 'se3_flow' / 'se3_best_checkpoint.pt'
if ranking_input is not None and se3_checkpoint.exists():
    run([
        'python', str(BASE_PATH / 'scripts' / '23_score_se3_geometry.py'),
        '--base', str(BASE_PATH),
        '--ranking-csv', str(ranking_input),
        '--checkpoint', str(se3_checkpoint),
        '--device', DEVICE,
    ])
else:
    print('Skipping SE(3) geometry scoring because ranking input or checkpoint is missing.')

run(['python', str(BASE_PATH / 'scripts' / '18_build_v5_3_benchmark_report.py'), '--base', str(BASE_PATH)])
run(['python', str(BASE_PATH / 'interface' / 'extract_data.py')], cwd=str(BASE_PATH))
run(['python', str(BASE_PATH / 'scripts' / '19_build_v5_3_gpu_decision_package.py'), '--base', str(BASE_PATH)])

artifact_zip = BASE_PATH / '09_reports' / 'kaggle_evidence_upgrade_outputs.zip'
artifact_zip.parent.mkdir(parents=True, exist_ok=True)
artifact_paths = [
    logged_generated, logged_attempt, logged_metrics, logged_summary, logged_novelty, logged_gap,
    diverse_generated, diverse_attempt, diverse_metrics, diverse_summary, diverse_novelty, diverse_gap,
    BASE_PATH / '06_docking' / 'v5_3_model_guided' / 'scores' / 'se3_geometry_scores.csv',
    BASE_PATH / '08_final_ranking' / 'v5_3_model_guided_pocket_electronic_ranked_candidates_with_se3.csv',
    BASE_PATH / '09_reports' / 'v5_3_benchmark' / 'EMD_V5_3_Benchmark_Report.md',
    BASE_PATH / '09_reports' / 'v5_3_benchmark' / 'emd_v5_3_benchmark_summary.json',
    BASE_PATH / 'interface' / 'data.js',
]
with zipfile.ZipFile(artifact_zip, 'w', compression=zipfile.ZIP_DEFLATED) as handle:
    for path in artifact_paths:
        if path.exists():
            handle.write(path, path.relative_to(BASE_PATH))
print('Artifact zip:', artifact_zip)